In [1]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24"
import os
import getpass

# 直接设置Hugging Face token，跳过登录界面
hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print("Hugging Face token set successfully!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your Hugging Face token:  ········


Hugging Face token set successfully!


In [2]:
# Step 2: Load FLARE-CAUSAL20-SC test set and preprocess
from datasets import load_dataset, Dataset

# FLARE-CAUSAL20-SC dataset
print("加载 FLARE-CAUSAL20-SC 数据集...")
ds_raw = load_dataset("ChanceFocus/flare-causal20-sc", split="test")
print(f"成功加载! 样本数: {len(ds_raw)}, 列: {ds_raw.column_names}")

# 定义标签
LABELS = ["noise", "causal"]

# 检查是否有标注
def _map_row(x):
    text = x.get("text") or x.get("sentence") or ""
    answer = x.get("answer", "")
    gold = x.get("gold", -1)
    choices = x.get("choices", LABELS)
    
    # 确定真实标签
    if gold >= 0 and gold < len(choices):
        true_label = choices[gold]
        has_label = True
    else:
        true_label = "unknown"
        has_label = False
    
    return {
        "text": text,
        "answer": answer,
        "gold": gold,
        "choices": choices,
        "label": true_label,
        "has_label": has_label,
        "labels": LABELS
    }

ds = Dataset.from_list([_map_row(r) for r in ds_raw])
bad = [i for i, r in enumerate(ds) if not r["text"]]
print(f"空文本样本数: {len(bad)}")

# 检查是否有标注
has_label_count = sum(1 for r in ds if r["has_label"])
print(f"有标注的样本数: {has_label_count}/{len(ds)}")

if has_label_count == 0:
    print("\n⚠️ 警告: 测试集没有标注，将只能进行推理测试，无法计算准确率")

# 显示第一个样本作为验证
print("\n第一个样本示例:")
print(f"文本: {ds[0]['text'][:200]}...")
print(f"标签: {ds[0]['label']}")
print(f"是否有标注: {ds[0]['has_label']}")

加载 FLARE-CAUSAL20-SC 数据集...
成功加载! 样本数: 8628, 列: ['id', 'query', 'answer', 'text', 'choices', 'gold']
空文本样本数: 0
有标注的样本数: 8628/8628

第一个样本示例:
文本:  Third Democratic presidential debate  September 12, 2019 at 9:54 PM EDT - Updated September 13 at 12:54 AM  WASHINGTON (AP)  -  Ten Democrats seeking the presidency tripped over some details Thursday...
标签: noise
是否有标注: True


In [3]:
# Step 3: Install dependencies, configure OpenAI, and record experiment metadata
%pip install -q "openai==1.40.2" "httpx==0.27.2" "httpcore==1.0.5" \
               "pandas>=2.2.2" "tqdm>=4.66.4" "requests>=2.31.0"

import os, getpass, json, time, platform
from importlib.metadata import version, PackageNotFoundError
import requests

# o3-mini配置
MODEL = "o3-mini"
BASE_URL = "https://api.openai.com/v1"

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# 文件命名
dataset_name = "flare_causal20_sc"
run_tag = f"{dataset_name}_{MODEL.replace('-', '_')}"
save_dir = os.getcwd()
pred_path = os.path.join(save_dir, f"{run_tag}_predictions.csv")
meta_path = os.path.join(save_dir, f"{run_tag}_metadata.json")

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

meta = {
    "dataset": "ChanceFocus/flare-causal20-sc",
    "split": "test",
    "labels": LABELS,
    "model": MODEL,
    "has_labels": has_label_count > 0,
    "labeled_samples": has_label_count,
    "openai_sdk": ver("openai"),
    "httpx": ver("httpx"),
    "httpcore": ver("httpcore"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "base_url": BASE_URL,
    "note": "o3-mini evaluation on FLARE-CAUSAL20-SC (Causal Classification)"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL, "| BASE_URL:", BASE_URL)
print(f"数据集是否有标注: {has_label_count > 0}")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your OpenAI API key:  ········


Meta saved -> C:\Users\klofu\flare_causal20_sc_o3_mini_metadata.json
MODEL: o3-mini | BASE_URL: https://api.openai.com/v1
数据集是否有标注: True


In [4]:
# Step 3.5: 测试50个样本并直接输出评估报告
print("="*60)
print("测试50个样本 - 直接输出评估报告")
print("="*60)

# 定义因果分类提示词函数
def _make_causal_prompt(text: str):
    return (
        "Task: Classify whether the following sentence indicates a causal relationship between events.\n\n"
        f"Text: {text}\n\n"
        "Instructions:\n"
        "1. Analyze if the text describes a cause-and-effect relationship\n"
        "2. Return ONLY one word: either 'causal' or 'noise'\n"
        "3. 'causal' if the text indicates a causal relationship\n"
        "4. 'noise' if it does not indicate causality\n\n"
        "Your response:"
    )

def parse_causal_response(response_text: str):
    """解析模型响应，返回 'causal' 或 'noise'"""
    response_text = response_text.strip().lower()
    
    if 'causal' in response_text and 'noise' not in response_text:
        return 'causal'
    elif 'noise' in response_text:
        return 'noise'
    else:
        return 'noise'

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def test_single_sample(text):
    """测试单个样本"""
    url = f"{BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    user_text = _make_causal_prompt(text)
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a financial NLP expert. Respond only with 'causal' or 'noise'."},
            {"role": "user", "content": user_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        
        if response.status_code != 200:
            return None, False, f"API Error: {response.status_code}"
            
        result = response.json()
        
        if 'choices' not in result or len(result['choices']) == 0:
            return None, False, "No response"
            
        txt = result['choices'][0]['message']['content']
        txt = _strip_code_fences(txt)
        pred_label = parse_causal_response(txt)
        return pred_label, True, None
        
    except Exception as e:
        return None, False, str(e)

# 测试前50个样本
print(f"\n开始测试前50个样本...\n")

test_size = min(50, len(ds))
test_results = []

for i in range(test_size):
    sample = ds[i]
    text = sample["text"]
    true_label = sample["label"] if sample["has_label"] else None
    
    pred_label, success, error = test_single_sample(text)
    
    test_results.append({
        "index": i,
        "text_preview": text[:80] + "..." if len(text) > 80 else text,
        "true_label": true_label,
        "pred_label": pred_label if success else "ERROR",
        "success": success,
        "error": error if not success else None
    })
    
    # 每10个样本打印一次进度
    if (i + 1) % 10 == 0:
        print(f"已完成: {i+1}/{test_size}")

# 输出结果表格
print("\n" + "="*80)
print("测试结果 (前50个样本)")
print("="*80)
print(f"{'序号':<6} {'预测标签':<12} {'真实标签':<12} {'状态':<8} {'文本预览'}")
print("-"*80)

for result in test_results:
    status = "✅" if result["success"] else "❌"
    true_display = result["true_label"] if result["true_label"] else "无标注"
    print(f"{result['index']:<6} {result['pred_label']:<12} {true_display:<12} {status:<8} {result['text_preview']}")

# 统计
success_count = sum(1 for r in test_results if r["success"])
causal_count = sum(1 for r in test_results if r["success"] and r["pred_label"] == "causal")
noise_count = sum(1 for r in test_results if r["success"] and r["pred_label"] == "noise")

print("-"*80)
print(f"\n统计:")
print(f"  成功调用: {success_count}/{test_size} ({success_count/test_size*100:.1f}%)")
print(f"  预测为 causal: {causal_count}")
print(f"  预测为 noise: {noise_count}")

# 如果有真实标签，计算评估指标
has_labels = any(r["true_label"] is not None for r in test_results)
if has_labels:
    # 只考虑有真实标签且成功的样本
    valid_results = [r for r in test_results if r["success"] and r["true_label"] is not None]
    
    if valid_results:
        from sklearn.metrics import accuracy_score, f1_score, classification_report
        
        y_true = [r["true_label"] for r in valid_results]
        y_pred = [r["pred_label"] for r in valid_results]
        
        accuracy = accuracy_score(y_true, y_pred)
        f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
        
        print(f"\n评估指标 (基于 {len(valid_results)} 个有标注样本):")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  F1-Macro: {f1_macro:.4f}")
        
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, zero_division=0))
        
        # 保存评估报告
        eval_results = {
            "model": MODEL,
            "dataset": "ChanceFocus/flare-causal20-sc",
            "test_mode": "50_samples_preview",
            "total_tested": test_size,
            "successful_calls": success_count,
            "labeled_samples": len(valid_results),
            "accuracy": float(accuracy),
            "f1_macro": float(f1_macro),
            "prediction_distribution": {
                "causal": causal_count,
                "noise": noise_count
            },
            "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())
        }
        
        eval_path = os.path.join(save_dir, f"{run_tag}_50samples_report.json")
        with open(eval_path, "w") as f:
            json.dump(eval_results, f, indent=2)
        print(f"\n评估报告已保存 -> {eval_path}")
    else:
        print("\n没有可评估的有标注样本")
else:
    print("\n注意: 测试集中没有真实标注，仅输出预测分布")

print("\n" + "="*80)
print("50样本测试完成")
print("="*80)

测试50个样本 - 直接输出评估报告

开始测试前50个样本...

已完成: 10/50
已完成: 20/50
已完成: 30/50
已完成: 40/50
已完成: 50/50

测试结果 (前50个样本)
序号     预测标签         真实标签         状态       文本预览
--------------------------------------------------------------------------------
0      causal       noise        ✅         Third Democratic presidential debate  September 12, 2019 at 9:54 PM EDT - Updat...
1      noise        noise        ✅         On the policy front, Bernie Sanders claimed his approach to health care has a s...
2      noise        noise        ✅         Joe Biden misrepresented recent history when he said the administration he serv...
3      causal       noise        ✅         Here's a look at some of the assertions in the third round of Democratic primar...
4      noise        noise        ✅         It killed 22 people, and injured many more, we were not defeated by that. Nor w...
5      causal       noise        ✅         JULIAN CASTRO, former U.S. housing secretary: Look, a few weeks ago a shooter d...
6      caus

Note: you may need to restart the kernel to use updated packages.
预测文件不存在: C:\Users\klofu\flare_causal20_sc_o3_mini_predictions.csv



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
